# 🧬 Dock-GPU | Rigid/Flexible Receptor Docking Pipeline

This notebook automates **GPU-accelerated molecular docking** using AutoDock-GPU on Google Colab. It is designed for users with **no coding background** — simply run each cell from top to bottom by clicking the ▶ play button.

---

### 📋 What you will need before starting:
- ✅ **Rigid/Flexible receptor files** (map files + `.pdbqt`) prepared using **ADFR Suite**
- ✅ **Ligand `.pdbqt` files** to be docked
- ✅ A docking shell script (e.g. `gpu_rigid_colab_docking.sh` or `gpu_flexible_colab_docking.sh`) configured for your system

---

> ⚠️ **Important:** Make sure your runtime is set to **GPU**. Go to `Runtime` → `Change runtime type` → Select `GPU`.

---

## 🖥️ Step 1 — System & GPU Verification
Run the cells below to confirm your GPU, CUDA, and compiler are ready. **Note the CUDA version** — you will need it in Step 2.

---

In [ ]:
#@title ▶ Check GPU Availability
#@markdown Displays the GPU assigned to your Colab session. Confirm a GPU is listed before proceeding.
!nvidia-smi

In [ ]:
#@title ▶ Check CUDA Version
#@markdown Shows the installed CUDA compiler version. **Note this version number** — it must match the version used in Step 2.
!nvcc --version

In [ ]:
#@title ▶ Check GCC Compiler Version
#@markdown Verifies that the GCC C compiler is available, which is required to build AutoDock-GPU.
!gcc --version

---
## ⚙️ Step 2 — Install & Compile AutoDock-GPU
This step downloads and compiles AutoDock-GPU using your system's CUDA installation.

> ⚠️ **Before running:** If the CUDA version shown above is **not** `12.8`, update the version number inside the cell below accordingly (e.g. change `cuda-12.8` to `cuda-12.4`). **Both the cell's CUDA version must be same.**

---

In [ ]:
#@title ▶ Download AutoDock-GPU
#@markdown Removes any previously downloaded version and fetches the latest AutoDock-GPU source code from the official GitHub repository.
! rm -r -f AutoDock-GPU; git clone https://github.com/ccsb-scripps/AutoDock-GPU.git

In [ ]:
#@title ▶ Compile AutoDock-GPU with CUDA
#@markdown Compiles AutoDock-GPU with GPU (CUDA) support using 256 work-items.
#@markdown
#@markdown > ⚠️ **The CUDA version below must match the version shown in Step 1 (nvcc --version).**
#@markdown > If they differ, update all occurrences of `cuda-12.8` to your installed version before running.
!export CUDAROOT=/usr/local/cuda-12.8;
!cd AutoDock-GPU; make DEVICE=CUDA NUMWI=256\
GPU_INCLUDE_PATH=/usr/local/cuda-12.8/include/ GPU_LIBRARY_PATH=/usr/local/cuda-12.8/lib64/

---
## 📁 Step 3 — Set Up Docking Directory Structure
Creates the folder layout required for organising your receptor and ligand files.

---

In [ ]:
#@title ▶ Create Docking Folder
#@markdown Creates the main `docking` directory that will hold all files for this session.
!mkdir docking

In [ ]:
#@title ▶ Verify Directory Was Created
#@markdown Lists all files and folders in the current directory. Confirm `docking` appears in the output.
!ls

In [ ]:
#@title ▶ Navigate into Docking Folder
#@markdown Moves into the `docking` directory so all subsequent files are organised within it.
%cd docking

---
## 🧪 Step 4 — Upload Receptor Files
Upload your rigid/flexible receptor files prepared with **ADFR Suite**.

> ✅ You should have map files + `.pdbqt` receptor file.

> 📌 **Uploading multiple receptors?** Set a new receptor name in the first cell below and **re-run both cells in this step** for each additional receptor zip. Every run deposits files into its own named subfolder — previously uploaded receptors are preserved.

---

In [ ]:
#@title ▶ Configure Receptor Directory & Upload Method
#@markdown ### 1️⃣ Enter your protein/receptor name:
#@markdown This will be the name of the subfolder inside `docking/rigid_flexible/`.
#@markdown Use underscores instead of spaces.

protein_name = "" #@param {type:"string"}

#@markdown ---
#@markdown ### 2️⃣ Choose how you want to upload your receptor files:
#@markdown | Method | Best For |
#@markdown |---|---|
#@markdown | **Individual File Picker** | First-time users, small number of files |
#@markdown | **Single ZIP Upload** | Faster uploads, all files pre-zipped on your PC |
#@markdown | **Google Drive** | Fastest — receptor already stored in Drive, reused across sessions |

upload_method = "Individual File Picker" #@param ["Individual File Picker", "Single ZIP Upload", "Google Drive"]

#@markdown ---
#@markdown *(For Google Drive only)* Paste the full path to your receptor folder in Drive:
drive_receptor_path = "/content/drive/MyDrive/your_receptor_folder" #@param {type:"string"}

import os
receptor_dir = f"/content/docking/rigid_flexible/{protein_name}"
os.makedirs(receptor_dir, exist_ok=True)
print(f"✅ Receptor directory created : {receptor_dir}")
print(f"📦 Upload method selected    : {upload_method}")
print("➡️  Run the next cell to upload your receptor files.")

In [ ]:
#@title ▶ Upload Receptor Files
#@markdown Uploads your receptor files using the method selected in the previous cell.
#@markdown All files will land directly inside `docking/rigid_flexible/<protein_name>/` — no subfolders.

import os, shutil, zipfile

receptor_dir = f"/content/docking/rigid_flexible/{protein_name}"

def _move_flat(src, dest_dir):
    """Move a file into dest_dir using only its basename — no subdirectory nesting."""
    shutil.move(src, os.path.join(dest_dir, os.path.basename(src)))

def _extract_flat(zip_path, dest_dir):
    """Extract all files from a ZIP directly into dest_dir, stripping any internal folder structure."""
    with zipfile.ZipFile(zip_path, 'r') as z:
        for member in z.infolist():
            fname = os.path.basename(member.filename)
            if not fname:          # skip directory entries inside the zip
                continue
            member.filename = fname
            z.extract(member, dest_dir)

def _copy_flat_from_drive(drive_path, dest_dir):
    """Copy only the files (no subfolders) from a Drive directory into dest_dir."""
    for fname in os.listdir(drive_path):
        src = os.path.join(drive_path, fname)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(dest_dir, fname))

# ── Method 1: Individual File Picker ──────────────────────────────────────────
if upload_method == "Individual File Picker":
    from google.colab import files
    print("📂 Upload all receptor files (map files + .pdbqt) using the file picker below.")
    print("   These should be prepared using ADFR Suite.\n")
    uploaded = files.upload()
    for filename in uploaded.keys():
        _move_flat(filename, receptor_dir)
    count = len(os.listdir(receptor_dir))
    print(f"\n✅ {count} file(s) now in: {receptor_dir}")

# ── Method 2: Single ZIP Upload ───────────────────────────────────────────────
elif upload_method == "Single ZIP Upload":
    from google.colab import files
    print("📦 Upload a single .zip containing all receptor files.")
    print("   Windows: select all → Right-click → Send to ZIP")
    print("   Mac    : select all → Right-click → Compress\n")
    uploaded = files.upload()
    for zip_filename in uploaded.keys():
        if zip_filename.endswith('.zip'):
            _extract_flat(zip_filename, receptor_dir)
            os.remove(zip_filename)
            count = len(os.listdir(receptor_dir))
            print(f"\n✅ ZIP extracted — {count} file(s) now in: {receptor_dir}")
        else:
            print(f"⚠️  '{zip_filename}' is not a .zip file. Please upload a valid ZIP archive.")

# ── Method 3: Google Drive ────────────────────────────────────────────────────
elif upload_method == "Google Drive":
    from google.colab import drive
    print("🔗 Mounting Google Drive — approve the permissions pop-up when it appears.")
    drive.mount('/content/drive', force_remount=False)
    if os.path.exists(drive_receptor_path):
        _copy_flat_from_drive(drive_receptor_path, receptor_dir)
        count = len(os.listdir(receptor_dir))
        print(f"\n✅ {count} receptor file(s) copied from Drive to: {receptor_dir}")
    else:
        print(f"\n❌ Path not found in Drive: {drive_receptor_path}")
        print("   Check the path entered in the previous cell and re-run.")

---
## 💊 Step 5 — Upload Ligand Files
Upload the ligand molecules you wish to dock. All files must be in **`.pdbqt` format**.

---

In [ ]:
#@title ▶ Configure Ligand Upload Method
#@markdown ### Choose how you want to upload your ligand files:
#@markdown All files must be in **`.pdbqt` format** and will be placed directly inside `docking/ligand/`.
#@markdown
#@markdown | Method | Best For |
#@markdown |---|---|
#@markdown | **Individual File Picker** | Small ligand batches, first-time use |
#@markdown | **Single ZIP Upload** | Faster uploads, all ligand `.pdbqt` files pre-zipped on your PC |
#@markdown | **Google Drive** | Fastest — ligand library already stored in Drive |

ligand_upload_method = "Individual File Picker" #@param ["Individual File Picker", "Single ZIP Upload", "Google Drive"]

#@markdown ---
#@markdown *(For Google Drive only)* Paste the full path to your ligand folder in Drive:
drive_ligand_path = "/content/drive/MyDrive/your_ligand_folder" #@param {type:"string"}

import os
os.makedirs("/content/docking/ligand", exist_ok=True)
print("✅ Ligand directory created : /content/docking/ligand")
print(f"📦 Upload method selected   : {ligand_upload_method}")
print("➡️  Run the next cell to upload your ligand files.")

In [ ]:
#@title ▶ Upload Ligand Files
#@markdown Uploads your ligand `.pdbqt` files using the method selected in the previous cell.
#@markdown All files will land directly inside `docking/ligand/` — no subfolders.

import os, shutil, zipfile

ligand_dir = "/content/docking/ligand"

def _move_flat(src, dest_dir):
    """Move a file into dest_dir using only its basename — no subdirectory nesting."""
    shutil.move(src, os.path.join(dest_dir, os.path.basename(src)))

def _extract_flat(zip_path, dest_dir):
    """Extract all files from a ZIP directly into dest_dir, stripping any internal folder structure."""
    with zipfile.ZipFile(zip_path, 'r') as z:
        for member in z.infolist():
            fname = os.path.basename(member.filename)
            if not fname:          # skip directory entries inside the zip
                continue
            member.filename = fname
            z.extract(member, dest_dir)

def _copy_flat_from_drive(drive_path, dest_dir):
    """Copy only the files (no subfolders) from a Drive directory into dest_dir."""
    for fname in os.listdir(drive_path):
        src = os.path.join(drive_path, fname)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(dest_dir, fname))

# ── Method 1: Individual File Picker ──────────────────────────────────────────
if ligand_upload_method == "Individual File Picker":
    from google.colab import files
    print("📂 Upload all ligand files in .pdbqt format using the file picker below.\n")
    uploaded = files.upload()
    for filename in uploaded.keys():
        _move_flat(filename, ligand_dir)
    count = len(os.listdir(ligand_dir))
    print(f"\n✅ {count} ligand file(s) now in: {ligand_dir}")

# ── Method 2: Single ZIP Upload ───────────────────────────────────────────────
elif ligand_upload_method == "Single ZIP Upload":
    from google.colab import files
    print("📦 Upload a single .zip containing all your ligand .pdbqt files.")
    print("   Windows: select all → Right-click → Send to ZIP")
    print("   Mac    : select all → Right-click → Compress\n")
    uploaded = files.upload()
    for zip_filename in uploaded.keys():
        if zip_filename.endswith('.zip'):
            _extract_flat(zip_filename, ligand_dir)
            os.remove(zip_filename)
            count = len(os.listdir(ligand_dir))
            print(f"\n✅ ZIP extracted — {count} ligand file(s) now in: {ligand_dir}")
        else:
            print(f"⚠️  '{zip_filename}' is not a .zip file. Please upload a valid ZIP archive.")

# ── Method 3: Google Drive ────────────────────────────────────────────────────
elif ligand_upload_method == "Google Drive":
    from google.colab import drive
    print("🔗 Mounting Google Drive — approve the permissions pop-up when it appears.")
    drive.mount('/content/drive', force_remount=False)
    if os.path.exists(drive_ligand_path):
        _copy_flat_from_drive(drive_ligand_path, ligand_dir)
        count = len(os.listdir(ligand_dir))
        print(f"\n✅ {count} ligand file(s) copied from Drive to: {ligand_dir}")
    else:
        print(f"\n❌ Path not found in Drive: {drive_ligand_path}")
        print("   Check the path entered in the previous cell and re-run.")

---
## 📜 Step 6 — Upload Docking Script
Upload the shell script (e.g. `gpu_rigid_colab_docking.sh` for rigid docking, or `gpu_flexible_colab_docking.sh` for flexible docking) that controls how AutoDock-GPU runs the docking job.

---

In [ ]:
#@title ▶ Upload Docking Shell Script
#@markdown A file picker will appear below. Select and upload your docking shell script
#@markdown (e.g. `gpu_rigid_colab_docking.sh` or `gpu_flexible_colab_docking.sh` — any `.sh` filename works).
#@markdown It will be placed directly inside the `docking` folder and remembered automatically
#@markdown for the remaining steps, so you never have to type the filename yourself.
from google.colab import files
import shutil, os

print("Upload your docking .sh script (rigid or flexible — any filename is accepted).")

uploaded = files.upload()

sh_files = [f for f in uploaded.keys() if f.lower().endswith(".sh")]
if not sh_files:
    raise FileNotFoundError("❌ No .sh file was uploaded. Please upload a docking shell script (.sh).")
if len(sh_files) > 1:
    print(f"⚠️ Multiple .sh files uploaded — using the first one: {sh_files[0]}")

SCRIPT_NAME = sh_files[0]

os.makedirs("/content/docking", exist_ok=True)
for filename in uploaded.keys():
    shutil.move(filename, f"/content/docking/{filename}")

# Persist the chosen script name to disk so later cells/sessions can recover it automatically
with open("/content/docking/.script_name", "w") as f:
    f.write(SCRIPT_NAME)

print(f"✅ Shell script uploaded successfully: {SCRIPT_NAME}")


---
## ✅ Step 7 — Verify File Counts
Before launching docking, confirm that all files were uploaded correctly.

---

In [ ]:
#@title ▶ List All Files in Docking Directory
#@markdown Displays all files present inside the `docking` folder. Confirm the shell script is visible.
!ls

In [ ]:
#@title ▶ Count Ligand Files
#@markdown Counts the total number of ligand files uploaded. This number determines how many docking runs will be performed.
!ls /content/docking/ligand | wc -l

In [ ]:
#@title ▶ Count Receptor Files
#@markdown Counts receptor files across **all** named receptor subfolders inside
#@markdown `docking/rigid_flexible/`. Each subfolder is listed individually so you
#@markdown can confirm every upload landed correctly before launching docking.
import os

base = "/content/docking/rigid_flexible"

if not os.path.isdir(base):
    print("⚠️  No receptor directory found. Run Step 4 first.")
else:
    subdirs = sorted([d for d in os.listdir(base)
                      if os.path.isdir(os.path.join(base, d))])
    if not subdirs:
        print("⚠️  No receptor subfolders found inside:", base)
    else:
        total = 0
        print(f"{'Receptor':<30}  {'Files':>5}")
        print("-" * 38)
        for sub in subdirs:
            path = os.path.join(base, sub)
            n = len([f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))])
            total += n
            print(f"  {sub:<28}  {n:>5}")
        print("-" * 38)
        print(f"  {'TOTAL':<28}  {total:>5}")
        print(f"\n✅ {len(subdirs)} receptor subfolder(s) ready for docking.")


---
## 🚀 Step 8 — Run AutoDock-GPU
Prepare and execute the docking script. This is the main computational step.

> ⏳ Docking time depends on the number of ligands and GPU speed. Large libraries may take a while — monitor the output below each cell.

---

In [ ]:
#@title ▶ Fix Script Line Endings
#@markdown Removes Windows-style line endings from the shell script to ensure it runs correctly on Linux (Colab).
#@markdown This step is necessary if the script was created or edited on a Windows machine.
#@markdown Works automatically with whichever script you uploaded in Step 6 (rigid or flexible).
import os, glob

# Recover SCRIPT_NAME if this cell is run in a fresh session (e.g. after a runtime restart)
try:
    SCRIPT_NAME
except NameError:
    name_file = "/content/docking/.script_name"
    if os.path.exists(name_file):
        SCRIPT_NAME = open(name_file).read().strip()
    else:
        candidates = glob.glob("/content/docking/*.sh")
        if not candidates:
            raise FileNotFoundError("❌ No docking .sh script found in /content/docking. Run Step 6 first.")
        SCRIPT_NAME = os.path.basename(candidates[0])

SCRIPT_PATH = f"/content/docking/{SCRIPT_NAME}"
print(f"Using docking script: {SCRIPT_NAME}")
!sed -i 's/\r$//' {SCRIPT_PATH}


In [ ]:
#@title ▶ Set Script Permissions
#@markdown Grants the system permission to execute the docking shell script.
import os, glob
try:
    SCRIPT_PATH
except NameError:
    name_file = "/content/docking/.script_name"
    if os.path.exists(name_file):
        SCRIPT_NAME = open(name_file).read().strip()
    else:
        candidates = glob.glob("/content/docking/*.sh")
        if not candidates:
            raise FileNotFoundError("❌ No docking .sh script found in /content/docking. Run Step 6 first.")
        SCRIPT_NAME = os.path.basename(candidates[0])
    SCRIPT_PATH = f"/content/docking/{SCRIPT_NAME}"
!chmod +x {SCRIPT_PATH}
print(f"✅ Execute permission set on: {SCRIPT_NAME}")


In [ ]:
#@title ▶ Run AutoDock-GPU Docking
#@markdown **Launches the GPU-accelerated docking job.** The output will display docking progress for each ligand.
#@markdown This cell may run for several minutes to hours depending on library size.
import os, glob
try:
    SCRIPT_PATH
except NameError:
    name_file = "/content/docking/.script_name"
    if os.path.exists(name_file):
        SCRIPT_NAME = open(name_file).read().strip()
    else:
        candidates = glob.glob("/content/docking/*.sh")
        if not candidates:
            raise FileNotFoundError("❌ No docking .sh script found in /content/docking. Run Step 6 first.")
        SCRIPT_NAME = os.path.basename(candidates[0])
    SCRIPT_PATH = f"/content/docking/{SCRIPT_NAME}"
!{SCRIPT_PATH}


---
## 📊 Step 9 — Verify & Download Results
Check that results were generated correctly, then download them to your computer.

> ✅ The number of result files should be **3× the number of ligand files** (each ligand produces 3 output files: `.dlg`, `.xml`, `.pdbqt`).

---

In [ ]:
#@title ▶ Count Docking Result Files
#@markdown Counts output files across **all** receptor result subfolders inside
#@markdown `docking/results_autodock_gpu/`. Each receptor subfolder is listed
#@markdown individually. Expected count per receptor: **3 × number of ligands**
#@markdown (`.dlg` + `.xml` + best-pose `.pdbqt` per ligand).
import os

results_base = "/content/docking/results_autodock_gpu"

if not os.path.isdir(results_base):
    print("⚠️  No results directory found. Run Step 8 first.")
else:
    # Exclude log/csv files at the root level — only count receptor subfolders
    subdirs = sorted([d for d in os.listdir(results_base)
                      if os.path.isdir(os.path.join(results_base, d))])
    if not subdirs:
        print("⚠️  No receptor result subfolders found inside:", results_base)
        root_files = [f for f in os.listdir(results_base)
                      if os.path.isfile(os.path.join(results_base, f))]
        if root_files:
            print(f"   ({len(root_files)} root-level file(s) present: log, csv, etc.)")
    else:
        total = 0
        print(f"{'Receptor':<30}  {'Result files':>12}")
        print("-" * 46)
        for sub in subdirs:
            path = os.path.join(results_base, sub)
            n = len([f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))])
            total += n
            print(f"  {sub:<28}  {n:>12}")
        print("-" * 46)
        print(f"  {'TOTAL':<28}  {total:>12}")
        print(f"\n✅ {len(subdirs)} receptor result subfolder(s) found.")
        print("   Expected per receptor: 3 × ligand count (.dlg + .xml + best-pose .pdbqt).")


In [ ]:
#@title ▶ Download Docking Results
#@markdown Compresses all docking results into a single `results.tar.gz` archive and initiates a download to your local computer.
#@markdown ⚠️ **Save this file before clearing — it contains all your docking outputs.**
!tar -czvf results.tar.gz -C /content/docking results_autodock_gpu
from google.colab import files
files.download("results.tar.gz")

---
## 🔁 Step 10 — Dock Another Batch of Ligands (Optional)
If you have more ligands to dock against the same receptor, use the cells below to clear the previous batch and start fresh.

> ⚠️ **Make sure you have already downloaded your results before running these cells.**
> After clearing, re-run **Step 5** (Create Ligand Directory + Upload Ligand Files) and then **Step 8** (Run Docking) again.

---

In [ ]:
#@title ▶ Clear Ligand Directory for Next Batch
#@markdown Deletes the current ligand folder to make room for a new batch of ligands.
#@markdown After running this cell, go back to **Step 5** to re-create the ligand folder and upload your next batch.
!rm -rf /content/docking/ligand

In [ ]:
#@title ▶ Clear Previous Results Directory
#@markdown Deletes the previous docking results to free up storage space before the next run.
#@markdown ⚠️ **Only run this after you have confirmed your results.tar.gz has been downloaded successfully.**
!rm -rf /content/docking/results_autodock_gpu

---
### 🎉 Docking Complete!
Your results archive (`results.tar.gz`) has been downloaded. Extract it on your local machine to access the individual `.dlg` output files for binding energy analysis.

---